In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Qubit 초기화

<details>
<summary><b>패키지 버전</b></summary>

이 페이지의 코드는 다음 요구 사항을 사용하여 개발되었습니다.
이 버전 이상을 사용하시기를 권장합니다.

```
qiskit-ibm-runtime~=0.43.1
```
</details>
Circuit이 IBM&reg; 양자 처리 장치(QPU)에서 실행될 때, Qubit이 0으로 초기화되도록 보장하기 위해 일반적으로 Circuit의 시작 부분에 암묵적인 초기화가 삽입됩니다. 이는 [프리미티브 실행 옵션](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-execution-options-v2)으로 설정된 `init_qubits` 플래그로 제어됩니다.

그러나 초기화 프로세스는 완벽하지 않아 상태 준비 오류가 발생할 수 있습니다. 오류를 완화하기 위해 시스템은 Circuit 사이에 반복 지연 시간(또는 `rep_delay`)도 삽입합니다. 각 Backend마다 기본 `rep_delay`가 다르지만, 환경이 Qubit을 초기화할 수 있도록 일반적으로 T1보다 깁니다. 기본 `rep_delay`는 `backend.default_rep_delay`를 실행하여 조회할 수 있습니다.

모든 IBM QPU는 동적 반복률 실행을 사용하며, 이를 통해 각 작업에 대한 `rep_delay`를 변경할 수 있습니다. 프리미티브 작업에서 제출하는 Circuit은 QPU에서 실행하기 위해 일괄 처리됩니다. 이 Circuit들은 요청된 각 샷에 대해 Circuit을 반복하여 실행됩니다. 실행은 다음 그림과 같이 Circuit과 샷의 행렬에서 열 방향으로 이루어집니다.

![첫 번째 열은 샷 0을 나타냅니다. Circuit은 0에서 3까지 순서대로 실행됩니다. 두 번째 열은 샷 1을 나타냅니다. Circuit은 0에서 3까지 순서대로 실행됩니다. 나머지 열도 같은 패턴을 따릅니다. ](../docs/images/guides/repetition-rate-execution/circuits_shots_matrix1.avif "열 방향 실행 행렬")

`rep_delay`는 Circuit 사이에 삽입되므로 실행의 각 샷에서 이 지연이 발생합니다. 따라서 `rep_delay`를 낮추면 총 QPU 실행 시간이 줄어들지만, 다음 이미지에서 볼 수 있듯이 상태 준비 오류율이 증가하는 대가를 치르게 됩니다.

![이 이미지는 `rep_delay` 값이 낮아질수록 상태 준비 오류율이 증가함을 보여줍니다.](../docs/images/guides/repetition-rate-execution/rep_delay.avif "반복 지연 대 오류율")

`rep_delay=0`과 `init_qubits=False`를 모두 설정하면 Qubit이 이전 샷의 최종 상태에서 시작되므로 Circuit이 사실상 "병합"됩니다.

프리미티브 작업의 Circuit들은 QPU 실행을 위해 일괄 처리되지만, PUB의 Circuit이 실행되는 순서는 보장되지 않습니다. 따라서 `pubs=[pub1, pub2]`를 제출하더라도 `pub1`의 Circuit이 `pub2`의 Circuit보다 먼저 실행된다는 보장은 없습니다. 또한 동일한 작업의 Circuit이 QPU에서 단일 배치로 실행된다는 보장도 없습니다.

## 프리미티브 작업에 rep_delay 지정하기

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

service = QiskitRuntimeService()

# Make sure your backend supports it
backend = service.least_busy(
    operational=True, min_num_qubits=100, dynamic_reprate_enabled=True
)

# Determine the allowable range
backend.rep_delay_range
sampler = Sampler(mode=backend)

# Specify a value in the supported range
sampler.options.execution.rep_delay = 0.0005

## 다음 단계
> **Tip:** - [양자 근사 최적화 알고리즘(QAOA)](/tutorials/quantum-approximate-optimization-algorithm) 튜토리얼에서 예제를 시도해 보세요.
>   - [프리미티브 시작하기](./get-started-with-primitives) 방법을 검토하세요.